In [2]:
import Pkg
Pkg.add("Polynomials")

    Updating registry at `C:\Users\dmitr\.julia\registries\General.toml`
     Project No packages added to or removed from `C:\Users\dmitr\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\dmitr\.julia\environments\v1.12\Manifest.toml`


In [78]:
using Polynomials

A = [1.0 2.0 3.0;
     2.0 4.0 5.0;
     3.0 5.0 6.0]

eps = 1e-6

n = size(A, 1)

# n = 3
# A = zeros(n, n)

# for i in 1:n
#     for j in i:n
#         value = round(1 + 9 * rand(), digits=2)
#         A[i,j] = value
#         A[j,i] = value
#     end
# end

println("A =")
display(A)

A =


3×3 Matrix{Float64}:
 1.0  2.0  3.0
 2.0  4.0  5.0
 3.0  5.0  6.0

In [79]:
E = zeros(n, n)
for i in 1:n
    E[i,i] = 1.0
end

D = copy(A)
B_total = copy(E)

for k in n:-1:2
    # step = n - k + 1
    B = copy(E)
    B_inv = copy(E)
    for j in 1:n
        if j == k-1
            B[k-1,j] = 1.0 / D[k,k-1]
        else
            B[k-1,j] = -D[k,j] / D[k,k-1]
        end
        B_inv[k-1,j] = D[k,j]
    end
    D = B_inv * D * B
    B_total = B_total * B
    # println("B$step=")
    # display(B)
    # println("B$step^(-1)=")
    # display(B_inv)
    # println("D$step=")
    # display(D)
end

P = D
p = collect(P[1,:])
println("\nМатрица Фробениуса:")
display(P)

println("\nКоэффициенты:")
for i in 1:n
    println("p$i=", p[i])
end

poly = Polynomial(vcat(-reverse(p), 1.0), :lambda)
println("\nХарактеристический многочлен:")
println(poly)


Матрица Фробениуса:


3×3 Matrix{Float64}:
 11.0  4.0  -1.0
  1.0  0.0   0.0
  0.0  1.0   0.0


Коэффициенты:
p1=11.000000000000002
p2=3.999999999999986
p3=-1.000000000000007

Характеристический многочлен:
1.0 - 4.0*lambda - 11.0*lambda^2 + 1.0*lambda^3


In [89]:
f(x) = poly(x)

lambda_lib = sort(real.(roots(poly)))

println("Библиотечные собственные значения:")
for i in 1:n
    println("lambda", i, " = ", lambda_lib[i])
end

Библиотечные собственные значения:
lambda1 = -0.5157294715892574
lambda2 = 0.17091518882718063
lambda3 = 11.34481428276208


In [90]:
left_bounds = Float64[]
right_bounds = Float64[]

println("\nКруги Гершгорина:")
for i in 1:n
    center = A[i,i]
    radius = 0.0
    for j in 1:n
        if i != j
            radius = radius + abs(A[i,j])
        end
    end
    a = center - radius
    b = center + radius
    push!(left_bounds, a)
    push!(right_bounds, b)
    println("Круг ", i)
    println("Центр: ", center)
    println("Радиус: ", radius)
    println("a = ", a)
    println("b = ", b)
    println()
end

g_left = minimum(left_bounds)
g_right = maximum(right_bounds)
println("Общие границы: [", g_left, ", ", g_right, "]")


Круги Гершгорина:
Круг 1
Центр: 1.0
Радиус: 5.0
a = -4.0
b = 6.0

Круг 2
Центр: 4.0
Радиус: 7.0
a = -3.0
b = 11.0

Круг 3
Центр: 6.0
Радиус: 8.0
a = -2.0
b = 14.0

Общие границы: [-4.0, 14.0]


In [91]:
intervals = Tuple{Float64,Float64}[]
parts = 10

for i in 1:parts
    a = g_left + (g_right - g_left)*(i-1)/parts
    b = g_left + (g_right - g_left)*i/parts
    if f(a)*f(b) < 0
        push!(intervals, (a, b))
    end
end

println("\nИнтервалы собственных значений:")
for i in 1:length(intervals)
    a, b = intervals[i]
    println("lambda", i, ": [", a, ", ", b, "]")
end


Интервалы собственных значений:
lambda1: [-2.2, -0.3999999999999999]
lambda2: [-0.3999999999999999, 1.4000000000000004]
lambda3: [10.4, 12.2]


In [92]:
function bisection(a, b)
    if f(a)*f(b) >= 0
        error("На концах интервала функция должна иметь разные знаки")
    end
    history = []
    n = 0
    while (b - a)/2 > eps
        c = (a + b)/2
        push!(history, (a, b, c))
        n = n + 1
        if f(c) == 0
            return c, n, history
        elseif f(a)*f(c) < 0
            b = c
        else
            a = c
        end
    end
    return (a + b)/2, n, history
end

b_values = Float64[]
b_iterations = Int[]
b_histories = []

println("\nМетод половинного деления:")
for i in 1:length(intervals)
    a, b = intervals[i]
    value, count, history = bisection(a, b)
    push!(b_values, value)
    push!(b_iterations, count)
    push!(b_histories, history)
    println("Собственное значение ", i, ": ", value)
    println("Количество итераций: ", count)
end


Метод половинного деления:
Собственное значение 1: -0.515729808807373
Количество итераций: 20
Собственное значение 2: 0.17091569900512715
Количество итераций: 20
Собственное значение 3: 11.34481477737427
Количество итераций: 20


In [93]:
tau = 0.005

function relaxation(x0, tau)
    history = [x0]
    n_iter = 0
    while abs(f(x0)) > eps
        x0 = x0 - tau*sign(df(x0))*f(x0)
        push!(history, x0)
        n_iter = n_iter + 1
        if n_iter > 1000
            error("Метод релаксаций не сошелся")
        end
    end
    return x0, n_iter, history
end

r_values = Float64[]
r_iterations = Int[]
r_histories = Vector{Float64}[]

println("\nМетод релаксаций:")

for i in 1:n
    a, b = intervals[i]
    x0 = (a + b)/2
    value, count, history = relaxation(x0, tau)
    push!(r_values, value)
    push!(r_iterations, count)
    push!(r_histories, history)
    println("Собственное значение ", i, ": ", value)
    println("Количество итераций: ", count)
end


Метод релаксаций:
Собственное значение 1: -0.5157295930065469
Количество итераций: 357
Собственное значение 2: 0.17091531463646048
Количество итераций: 368
Собственное значение 3: 11.344814278935731
Количество итераций: 15


In [94]:
function secant(x0, x1)
    history = [x0, x1]
    n = 0
    while abs(f(x1)) > eps
        x_new = x1 - f(x1)*(x1 - x0)/(f(x1) - f(x0))
        push!(history, x_new)
        x0 = x1
        x1 = x_new
        n = n + 1
    end
    return x1, n, history
end

s_values = Float64[]
s_iterations = Int[]
s_histories = Vector{Float64}[]

println("\nМетод секущих:")
for i in 1:length(intervals)
    a, b = intervals[i]
    value, count, history = secant((a+b)/2, b)
    push!(s_values, value)
    push!(s_iterations, count)
    push!(s_histories, history)
    println("Собственное значение ", i, ": ", value)
    println("Количество итераций: ", count)
end


Метод секущих:
Собственное значение 1: -0.5157294715760158
Количество итераций: 6
Собственное значение 2: 0.17091525255977116
Количество итераций: 6
Собственное значение 3: 11.34481428264443
Количество итераций: 4


In [95]:
println("\nСравнение с библиотечным решением:")

for i in 1:n
    println("\nlambda", i, ":")
    println("Точное решение: ", lambda_lib[i])
    println("Метод половинного деления:")
    println("Корень: ", b_values[i])
    println("Сравнение с точным: ", abs(lambda_lib[i] - b_values[i]))
    println("Метод релаксаций:")
    println("Корень: ", r_values[i])
    println("Сравнение с точным: ", abs(lambda_lib[i] - r_values[i]))
    println("Метод секущих:")
    println("Корень: ", s_values[i])
    println("Сравнение с точным: ", abs(lambda_lib[i] - s_values[i]))
end


Сравнение с библиотечным решением:

lambda1:
Точное решение: -0.5157294715892574
Метод половинного деления:
Корень: -0.515729808807373
Сравнение с точным: 3.37218115542548e-7
Метод релаксаций:
Корень: -0.5157295930065469
Сравнение с точным: 1.21417289444814e-7
Метод секущих:
Корень: -0.5157294715760158
Сравнение с точным: 1.3241630014704242e-11

lambda2:
Точное решение: 0.17091518882718063
Метод половинного деления:
Корень: 0.17091569900512715
Сравнение с точным: 5.101779465277456e-7
Метод релаксаций:
Корень: 0.17091531463646048
Сравнение с точным: 1.258092798550603e-7
Метод секущих:
Корень: 0.17091525255977116
Сравнение с точным: 6.373259053837899e-8

lambda3:
Точное решение: 11.34481428276208
Метод половинного деления:
Корень: 11.34481477737427
Сравнение с точным: 4.946121894278122e-7
Метод релаксаций:
Корень: 11.344814278935731
Сравнение с точным: 3.826349015412234e-9
Метод секущих:
Корень: 11.34481428264443
Сравнение с точным: 1.17649889830318e-10
